# 05. Streamlit · FastAPI · n8n 예측 준비

`04_model_tuning_final.ipynb`에서 저장한 모델 자산을 다시 불러와
실제 서비스에서 사용할 수 있는지 검증합니다.

원본 데이터는 필요하지 않으며 `models/`만 있으면 실행할 수 있습니다.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    if current.name == "notebooks":
        return current.parent
    if (current / "notebooks").exists():
        return current
    return current.parent

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [INTERIM_DIR, PROCESSED_DIR, SAMPLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

model = joblib.load(MODEL_DIR / "best_tuned_model.pkl")
threshold = float(joblib.load(MODEL_DIR / "best_tuned_threshold.pkl"))

with (MODEL_DIR / "feature_cols.json").open(encoding="utf-8") as f:
    feature_cols = json.load(f)

with (MODEL_DIR / "feature_medians.json").open(encoding="utf-8") as f:
    feature_medians = json.load(f)

model_feature_cols = [str(c) for c in model.get_booster().feature_names]

assert feature_cols == model_feature_cols
print("모델:", type(model).__name__)
print("Threshold:", threshold)
print("피처 수:", len(feature_cols))

In [ ]:
EPS = 1e-6

BASE_FEATURES = [
    "avg_stay_hour", "avg_daily_enter", "visit_days", "first_visit_delay",
    "consecutive_group_2일", "consecutive_group_3일", "first_visit_hour",
    "n_sites_visited", "area_pyeong", "is_post_covid",
]

def make_features(input_df):
    df = input_df.copy()

    missing = [c for c in BASE_FEATURES if c not in df.columns]
    if missing:
        raise ValueError("누락 컬럼: " + ", ".join(missing))

    for c in BASE_FEATURES:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    trial_date = pd.to_datetime(df.get("trial_date"), errors="coerce")
    if trial_date is None:
        df["trial_month"] = 1
        df["trial_dayofweek"] = 0
    else:
        df["trial_month"] = trial_date.dt.month.fillna(1).astype(int)
        df["trial_dayofweek"] = trial_date.dt.dayofweek.fillna(0).astype(int)

    df["is_weekend_trial"] = (df["trial_dayofweek"] >= 5).astype(int)

    df["stay_x_visit"] = df["avg_stay_hour"] * df["visit_days"]
    df["enter_x_visit"] = df["avg_daily_enter"] * df["visit_days"]
    df["stay_x_enter"] = df["avg_stay_hour"] * df["avg_daily_enter"]
    df["enter_per_visit_day"] = df["avg_daily_enter"] / (df["visit_days"] + EPS)
    df["stay_per_enter"] = df["avg_stay_hour"] / (df["avg_daily_enter"] + EPS)
    df["delay_per_visit_day"] = df["first_visit_delay"] / (df["visit_days"] + EPS)
    df["visit_delay_interaction"] = df["visit_days"] * df["first_visit_delay"]

    df["is_fast_visit"] = (df["first_visit_delay"] <= 1).astype(int)
    df["is_delayed_visit"] = (df["first_visit_delay"] >= 3).astype(int)
    df["is_frequent_user"] = (df["avg_daily_enter"] >= 2).astype(int)
    df["is_long_stay"] = (df["avg_stay_hour"] >= 2).astype(int)
    df["is_short_frequent"] = (
        (df["avg_stay_hour"] < 1) & (df["avg_daily_enter"] >= 2)
    ).astype(int)
    df["is_long_frequent"] = (
        (df["avg_stay_hour"] >= 2) & (df["avg_daily_enter"] >= 2)
    ).astype(int)

    df["is_morning"] = df["first_visit_hour"].between(6, 11).astype(int)
    df["is_afternoon"] = df["first_visit_hour"].between(12, 17).astype(int)
    df["is_evening"] = df["first_visit_hour"].between(18, 23).astype(int)

    df["first_visit_hour_sin"] = np.sin(2*np.pi*df["first_visit_hour"]/24)
    df["first_visit_hour_cos"] = np.cos(2*np.pi*df["first_visit_hour"]/24)
    df["trial_month_sin"] = np.sin(2*np.pi*df["trial_month"]/12)
    df["trial_month_cos"] = np.cos(2*np.pi*df["trial_month"]/12)

    df["post_covid_x_visit_days"] = df["is_post_covid"] * df["visit_days"]
    df["post_covid_x_avg_stay"] = df["is_post_covid"] * df["avg_stay_hour"]
    df["post_covid_x_daily_enter"] = df["is_post_covid"] * df["avg_daily_enter"]

    for c in [
        "avg_stay_hour", "avg_daily_enter", "first_visit_delay",
        "area_pyeong", "n_sites_visited", "stay_x_visit",
    ]:
        df[f"log1p_{c}"] = np.log1p(df[c].clip(lower=0))

    X = df[feature_cols].copy().apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)

    for c in feature_cols:
        X[c] = X[c].fillna(float(feature_medians[c]))

    return X

In [ ]:
sample = pd.DataFrame([
    {
        "customer_id": "DEMO001",
        "trial_date": "2023-08-21",
        "avg_stay_hour": 1.2,
        "avg_daily_enter": 3.0,
        "visit_days": 3,
        "first_visit_delay": 0,
        "consecutive_group_2일": 1,
        "consecutive_group_3일": 1,
        "first_visit_hour": 9,
        "n_sites_visited": 2,
        "area_pyeong": 100,
        "is_post_covid": 1,
    }
])

X_sample = make_features(sample)
probability = model.predict_proba(X_sample)[:, 1]
prediction = (probability >= threshold).astype(int)

result = sample.copy()
result["payment_probability"] = probability
result["prediction"] = prediction
result["prediction_label"] = np.where(prediction == 1, "결제 예상", "미결제 예상")

result[["customer_id", "payment_probability", "prediction_label"]]

In [ ]:
output_path = SAMPLE_DIR / "prediction_result_sample.csv"
result.to_csv(output_path, index=False, encoding="utf-8-sig")
print("저장 완료:", output_path)